In [1]:
import numpy as np
import xarray as xr
import numba as nb
import pandas as pd
from datetime import datetime
from functools import partial
import os
from multiprocessing import Manager
import sys
sys.path.insert(0,'/cluster/work/climate/dnikolo/n2o')
from glaciation_time_estimator.auxiliary_func.config_reader import read_config
# from glaciation_time_estimator.data_postprocessing.Single_cloud_analysis import Cloud
from glaciation_time_estimator.data_postprocessing.Job_result_fp_generator import generate_tracking_filenames
from glaciation_time_estimator.auxiliary_func.Nestable_multiprocessing import NestablePool
import datetime as dt

In [2]:
import numpy as np
import xarray as xr
from numba import njit, typed, types
import pandas as pd
from datetime import datetime
sys.path.insert(0,'/cluster/work/climate/dnikolo/n2o')
from glaciation_time_estimator.auxiliary_func.config_reader import read_config
# from glaciation_time_estimator.data_postprocessing.Single_cloud_analysis import Cloud
from glaciation_time_estimator.data_postprocessing.Job_result_fp_generator import generate_tracking_filenames
from glaciation_time_estimator.data_postprocessing.dardar_reindexing import build_dardar_index, match_dardar_to_cloud
from multiprocessing import Manager, Pool
from glaciation_time_estimator.auxiliary_func.Nestable_multiprocessing import NestablePool
from functools import partial
import os
# from memory_profiler import profile





In [15]:
# I have no idea why this sqrt but will keep it to be sure
from math import sqrt as m_sqrt
class Cloud:
    """
    A class containing all the information of a given tracked feature (cloud)
    """
    # def __new__(self, *args, **kwargs):
    #     return super().__new__(self)
    def __init__(self, cloud_id, is_resampled):
        # Set initial parameters
        self.id = cloud_id # tracknumber
        self.is_resampled = is_resampled # whether we are analyzing output directly from CLAAS or after resampling
        self.crit_fraction = 0.1 # For simplified analysis. Critical IF below which the cloud is considered liquid. Above 1-crit_fraction the cloud is considered ice. In the medium range the cloud is considered mixed.
        # Bools inidicating if the cloud has been liquid at any point
        self.is_liq: bool = False # Has the cloud been liquid at any point?
        self.is_mix: bool = False # Has the cloud been mixed at any point?
        self.is_ice: bool = False # Has the cloud been ice at any point?
        # Max and min cloud size in pixels
        self.max_size_km: float = 0.0 # Maximum area of the cloud in km
        self.max_size_px: int = 0 # Maximum area of the cloud in num. of pixels
        self.min_size_km: float = 510.0e6 # Minimum area of the cloud in km. Start with really large value and minimize from there 
        self.min_size_px: int = 3717*3717 # Minimum area of the cloud in num. of pixels. 

        # Variables giving the first and last 4 timesteps (1 hour) of the cloud ice fraction - both arrays run in the same time direction start: [1 , 2 , 3 , 4] ... end: [1 , 2 , 3 , 4]
        self.start_ice_fraction_arr = np.empty(4)
        self.end_ice_fraction_arr = np.empty(4)
        # self.ice_fraction_arr=np.empty(max_timesteps)
        self.ice_fraction_list = []

        self.max_water_fraction: float = 0.0
        self.max_ice_fraction: float = 0.0

        self.track_start_time: dt.datetime = None
        self.track_end_time: dt.datetime = None
        self.track_length = None

        self.glaciation_start_time: dt.datetime = None
        self.glaciation_end_time: dt.datetime = None

        self.n_timesteps = None

        self.sum_cloud_cot = 0
        self.avg_cot = None
        self.cot_timestep_counter = 0
        self.mean_cot_list = []
        self.std_cot_list = []

        self.sum_cloud_ctp = 0
        self.avg_ctp = None
        self.ctp_timestep_counter = 0
        self.mean_ctp_list = []
        self.std_ctp_list = []

        self.sum_cloud_ctt = 0
        self.avg_ctt = None
        self.mean_ctt_list = []
        self.std_ctt_list = []

        self.sum_cloud_cwp = 0
        self.avg_cwp = None
        self.cwp_timestep_counter = 0
        self.mean_cwp_list = []
        self.std_cwp_list = []

        self.sum_cloud_lat = 0.0
        self.sum_cloud_lon = 0.0
        self.avg_cloud_lat = None
        self.avg_cloud_lon = None
        self.lon_list = []
        self.lat_list = []

        self.sum_cloud_size_km = 0.0
        self.avg_cloud_size_km = None
        self.cloud_size_km_list = []
        self.large_pixel_cloud = False

        self.sum_cloud_size_px = 0.0
        self.avg_cloud_size_px = None

        self.valid_cot_cloud = False
        self.cot_nan_frac_list = []

        self.valid_ctp_cloud = False
        self.ctp_nan_frac_list = []

        self.n_timesteps_no_cloud = 0
        self.terminate_cloud = False
        
        # dd_* = DARDAR variables
        self.dd_cph_list = []
        self.dd_cth_list = []
        self.dd_cth_std_list = []

        self.dd_cph_deviation = []
        self.dd_cph_std_list = []
        self.dd_pix_claas_cph_meas = []


    def __str__(self):
        return f"{self.is_liq},{self.is_mix},{self.is_ice},"
    # In resampled clouds pixel area should be the area in degrees lon_resolution*lat_resolution

    def weighted_avg_and_std(self, values, weights):
        """
        Return the weighted average and standard deviation.

        They weights are in effect first normalized so that they 
        sum to 1 (and so they must not all be 0).

        values, weights -- NumPy ndarrays with the same shape.
        """
        average = np.average(values, weights=weights)
        # Fast and numerically precise:
        variance = np.average((values-average)**2, weights=weights)
        return (average, m_sqrt(variance))

    def check_status_inputs(self, cloud_values,pixel_area_non_agg, pixel_area_agg):
        """
        Do checks on the validity of some inputs.
        """
        assert (~np.isnan(pixel_area_non_agg).any()), "NaN values in pixel area array after filtering"
        assert (~np.isnan(pixel_area_agg).any()).any(), "NaN values in pixel area array after filtering"
        assert (pixel_area_agg > 0).all(), f"0 or negative values in aggregated pixel area array {pixel_area_agg}"
        assert (len(pixel_area_agg) == len(cloud_values)), f"Length of pixel area array ({len(pixel_area_non_agg)}) does not match length of cloud values array ({len(cloud_values)})"



    def dardar_get_valid_values(self,dd_cph,dd_cth,dd_cth_std):
        """
        Get valid DARDAR values for the cloud. This is used for validation of the cloud properties.
        """
        if dd_cph is not None:
            dd_cph = dd_cph[~np.isnan(dd_cph)]
        if dd_cth is not None:
            dd_cth = dd_cth[~np.isnan(dd_cth)]
        if dd_cth_std is not None:
            dd_cth_std = dd_cth_std[~np.isnan(dd_cth_std)]
        return dd_cph, dd_cth, dd_cth_std
    
    def check_non_agg_values(self, cot_values, ctp_values, ctt_values, cloud_lat, cloud_lon, pixel_area_non_agg):
        """Check the validity of the cloud values and filter out invalid pixels. This is important to avoid errors in
        """
        ## Aggregated pixels may cover NaN (e.g. outside the globe) in the original resolution.
        ## We filter those using ind_to_take
        ind_to_take = ~np.isnan(pixel_area_non_agg)
        pixel_area_non_agg = pixel_area_non_agg[ind_to_take]
        
            
        if sum(pixel_area_non_agg) == 0 or len(pixel_area_non_agg)==0:
            message = f"""
            Cloud_properties:\n
            ID: {self.id}\n
            Time: {time}\n
            Max_size_km: {self.max_size_km}\n
            Valid_cot_cloud: {self.valid_cot_cloud}\n
            Valid_ctp_cloud: {self.valid_ctp_cloud}\n
            pixel_area_non_agg: {pixel_area_non_agg}\n
            Ind to take: {ind_to_take}\n
            Cloud_values: {cloud_values}\n
            Cot_values: {cot_values}\n
            Ctp_values: {ctp_values}\n
            Cloud_lat: {cloud_lat}\n
            Cloud_lon: {cloud_lon}
            """ 
            raise ValueError("All pixel areas are zero or pixel area size is 0:\n"+message)
        # print(message)

        cot_values = cot_values[ind_to_take] if cot_values.size !=0 else None
        ctp_values = ctp_values[ind_to_take] if ctp_values.size !=0 else None
        ctt_values = ctt_values[ind_to_take] if ctt_values.size !=0 else None
        cloud_lat = cloud_lat[ind_to_take]
        cloud_lon = cloud_lon[ind_to_take]
        return cot_values, ctp_values, ctt_values, cloud_lat, cloud_lon, pixel_area_non_agg

    def update_status(self, time: dt.datetime, cloud_values: np.array, cot_values, ctp_values, ctt_values, cloud_lat, cloud_lon, pixel_area_non_agg, pixel_area_agg, dd_cph=None, dd_cth=None, dd_cth_std=None):
        """
        Main function of the class. This is executed each timestep the cloud is present.
        Data about the cloud top pixels is passed and the cloud properties and time series are updated.
        It is important that this function is called sequentially

        Inputs:
        --------
        time: datetime
            The time of the current timestep
        cloud_values: np.array
            1d Array of cloud phase values for the cloud top pixels
        cot_values: np.array
            Array of cloud optical thickness values for the cloud top pixels
        ctp_values: np.array
            Array of cloud top pressure values for the cloud top pixels
        cloud_lat: np.array 
            Array of latitudes for the cloud top pixels
        cloud_lon: np.array
            Array of longitudes for the cloud top pixels
        pixel_area_non_agg: np.array
            Array of pixel areas for the cloud top pixels (non-aggregated)
        pixel_area_agg: np.array
            Array of pixel areas for the cloud top pixels (aggregated)
        
        All the arrays must be the same length with corresponding values at the same indices.
        
        Outputs:
           None
        """


        ## Aggregated pixels may cover NaN (e.g. outside the globe) in the original resolution.
        ## We filter those using check_non_agg_values
        cot_values, ctp_values, ctt_values, cloud_lat, cloud_lon, pixel_area_non_agg = self.check_non_agg_values(cot_values, ctp_values, ctt_values, cloud_lat, cloud_lon, pixel_area_non_agg)
        
        cloud_size_px = cloud_values.shape[0]

        # We calculated weighted average position of the cloud in latitude and longitude
        if not self.is_resampled:
            cloud_lat = np.average(cloud_lat, weights=pixel_area_non_agg)
            cloud_lon = np.average(cloud_lon, weights=pixel_area_non_agg)
            # cloud_lat = 10
            # cloud_lon = 10
        else:
            cloud_lat = np.average(cloud_lat, weights=pixel_area_non_agg)
            cloud_lon = np.average(cloud_lon, weights=pixel_area_non_agg)
        # print(cloud_values)

        
        if cloud_size_px:
            self.n_timesteps_no_cloud = 0
            valid_values = cloud_values[cloud_values >= 1-1e3] - 1
            agg_area_weights = pixel_area_agg[cloud_values >= 1-1e3]
            # print(len(valid_values)/len(cloud_values))
            # print("Agg_area_weights:", agg_area_weights)
            try:
                ice_fraction = np.average(valid_values, weights=agg_area_weights)
            except Exception as e:
                message_2 = f"agg_area_weights: {agg_area_weights}\n valid_values: {valid_values}\n cloud_values: {cloud_values}\n pixel_area_non_agg: {pixel_area_non_agg}\n pixel_area_agg: {pixel_area_agg}"
                raise ValueError("Error calculating ice fraction with message:\n" + message + message_2 + e)
            # print(valid_values)
            # ice_fraction=float(np.count_nonzero(cloud_values==2))/float(cloud_size_px)
            water_fraction = 1-ice_fraction
            # assert math.isclose(water_fraction+ice_fraction,1)
            # print(water_fraction)
            # print(water_fraction)f cloud_arr[track_number-1] is None:

            if not (self.track_start_time):
                self.track_start_time = time
                self.n_timesteps = 1
            else:
                self.n_timesteps += 1
            if self.n_timesteps <= 4:
                self.start_ice_fraction_arr[self.n_timesteps-1] = ice_fraction
            # Check and set type of cloud
            if water_fraction > 1-self.crit_fraction:
                self.is_liq = True
            elif water_fraction > self.crit_fraction:
                self.is_mix = True
            else:
                self.is_ice = True
            # if self.is_resampled:
            #     cloud_size_km = sum(pixel_area_non_agg*cloud_size_px * \
            #         np.cos(np.deg2rad(cloud_lat))*111.321*111.111)
            # else:
            cloud_size_km = pixel_area_non_agg.sum()
            large_pixel_frac = np.count_nonzero(
                pixel_area_non_agg > 66)/pixel_area_non_agg.shape[0]
            if large_pixel_frac > 0.1 or pixel_area_non_agg.max() > 110:
                self.large_pixel_cloud = True
            self.cloud_size_km_list.append(cloud_size_km)
            self.max_size_km = max(self.max_size_km, cloud_size_km)
            self.min_size_km = min(self.min_size_km, cloud_size_km)

            self.max_size_px = max(self.max_size_px, cloud_size_px)
            self.min_size_px = min(self.min_size_px, cloud_size_px)

            self.sum_cloud_size_px += cloud_size_px
            self.avg_cloud_size_px = self.sum_cloud_size_px/self.n_timesteps

            self.sum_cloud_size_km += cloud_size_km
            self.avg_cloud_size_km = self.sum_cloud_size_km/self.n_timesteps

            # I assume that water_frac+ice_frac=1

            self.max_water_fraction = max(
                self.max_water_fraction, water_fraction)
            self.max_ice_fraction = max(
                self.max_ice_fraction, 1-water_fraction)

            self.sum_cloud_lat += cloud_lat
            self.sum_cloud_lon += cloud_lon
            self.lon_list.append(cloud_lon)
            self.lat_list.append(cloud_lat)
            self.avg_cloud_lat = self.sum_cloud_lat/self.n_timesteps
            self.avg_cloud_lon = self.sum_cloud_lon/self.n_timesteps

            self.track_end_time = time
            self.track_length = self.track_end_time-self.track_start_time

            self.end_ice_fraction_arr[0:3] = self.end_ice_fraction_arr[1:4]
            self.end_ice_fraction_arr[3] = ice_fraction

            # self.ice_fraction_arr[n_timesteps]=ice_fraction
            self.ice_fraction_list.append(ice_fraction)
            if cot_values is not None:
                self.update_cot_variables(cot_values, pixel_area_non_agg)
            if ctp_values is not None:
                self.update_ctp_variables(ctp_values, pixel_area_non_agg)
            if ctt_values is not None:
                self.update_ctt_variables(ctt_values, pixel_area_non_agg)
            
            # dd_cph, dd_cth, dd_cth_std = self.dardar_get_valid_values(dd_cph, dd_cth, dd_cth_std)
            if dd_cph is not None or dd_cth is not None or dd_cth_std is not None:
                self.update_dardar_variables(dd_cph,dd_cth,dd_cth_std, cloud_values, cloud_lat)
            
    def update_dardar_variables(self, dd_cph,dd_cth,dd_cth_std, cloud_values, cloud_lat):
        self.update_dardar_cph( dd_cph, cloud_values, cloud_lat)
        
    def update_dardar_cph(self, dd_cph, cloud_values, cloud_lat ):
        if dd_cph.size > 0:
            # 0.99 is 1 with accounting for floating point errors
            dd_cloudy_pixels = dd_cph >= 0.99
            dd_valid_pixels = dd_cph >=0
            dd_cph_cloudy = dd_cph[dd_cloudy_pixels]-1
            
            if dd_cph_cloudy.size > 0:
                print("Checking dardar cloud")
                if dd_valid_pixels.shape!=cloud_values.shape:
                    raise Exception(f"Mismatch between dardar values size and cloud values size dd_cloudy_pixels:\n {dd_cloudy_pixels}\n cloud_values: {cloud_values}\n cloud lat {cloud_lat}")
                dd_pixel_IF_claas_measurement = cloud_values[dd_valid_pixels]
                dd_cph_measured = dd_cph_cloudy.mean()
                self.dd_cph_list.append(dd_cph_measured)
                self.dd_cph_deviation.append(dd_pixel_IF_claas_measurement - dd_cph_measured)
                self.dd_cph_std_list.append(dd_cph_cloudy.std())
                self.dd_pix_claas_cph_meas.append(dd_pixel_IF_claas_measurement)
            else:
                # in case there are no cloudy pixels
                self.dd_cph_list.append(-1)
                self.dd_cph_deviation.append(-99)
                self.dd_cph_std_list.append(-1)
                self.dd_pix_claas_cph_meas.append(-1)
        else:
            self.dd_cph_list.append(np.nan)
            self.dd_cph_deviation.append(np.nan)
            self.dd_cph_std_list.append(np.nan)


            
    def update_cot_variables(self, cot_values, pixel_area_non_agg):
        cot_nan_frac = np.count_nonzero(
            np.isnan(cot_values))/cot_values.shape[0]
        if cot_nan_frac > 0.1:
            self.valid_cot_cloud = False
        self.cot_nan_frac_list.append(cot_nan_frac)
        weights = pixel_area_non_agg[~np.isnan(cot_values)]
        if len(weights) > 0:
            cot_values = cot_values[~np.isnan(cot_values)]
            mean_cot, std_cot = self.weighted_avg_and_std(cot_values, weights)
            if cot_nan_frac < 0.1:
                self.sum_cloud_cot += mean_cot
                self.cot_timestep_counter += 1
                self.avg_cot = self.sum_cloud_cot/self.cot_timestep_counter
        else:
            mean_cot = np.nan
            std_cot = np.nan
        self.mean_cot_list.append(mean_cot)
        self.std_cot_list.append(std_cot)

    def update_ctp_variables(self, ctp_values, pixel_area_non_agg):
        ctp_nan_frac = np.count_nonzero(
            np.isnan(ctp_values))/ctp_values.shape[0]
        if ctp_nan_frac > 0.1:
            self.valid_ctp_cloud = False
        self.ctp_nan_frac_list.append(ctp_nan_frac)
        weights = pixel_area_non_agg[~np.isnan(ctp_values)]
        if len(weights) > 0:
            ctp_values = ctp_values[~np.isnan(ctp_values)]
            mean_ctp, std_ctp = self.weighted_avg_and_std(ctp_values, weights)
            if ctp_nan_frac < 0.1:
                self.sum_cloud_ctp += mean_ctp
                self.ctp_timestep_counter += 1
                self.avg_ctp = self.sum_cloud_ctp/self.ctp_timestep_counter
        else:
            mean_ctp = np.nan
            std_ctp = np.nan
        self.mean_ctp_list.append(mean_ctp)
        self.std_ctp_list.append(std_ctp)

    def update_ctt_variables(self, ctt_values, pixel_area_non_agg):
        weights = pixel_area_non_agg[~np.isnan(ctt_values)]
        if len(weights) > 0:
            ctt_values = ctt_values[~np.isnan(ctt_values)]
            mean_ctt, std_ctt = self.weighted_avg_and_std(ctt_values, weights)
            self.sum_cloud_ctt += mean_ctt
            self.avg_ctt = self.sum_cloud_ctt/self.n_timesteps
        else:
            mean_ctt = np.nan
            std_ctt = np.nan
        self.mean_ctt_list.append(mean_ctt)
        self.std_ctt_list.append(std_ctt)

    def update_missing_cloud(self):
        if self.track_end_time and (not self.terminate_cloud):
            self.n_timesteps_no_cloud += 1
            if self.n_timesteps_no_cloud > 1:
                self.terminate_cloud = True


## Current tested version

In [16]:
# ---------- helper Numba types ----------
coord_type = types.UniTuple(types.int16, 2)        # (row, col)
list_type = types.ListType(coord_type)            # list of coordinates
array2d_type = types.int16[:, ::1]                   # (2, n_pts) C-contiguous
dict_lists_t = types.DictType(types.int64, list_type)
dict_arrays_t = types.DictType(types.int64, array2d_type)


@njit
def extract_cloud_coordinates(cloudtracknumber_field,   # 3-D, shape (1, ny, nx)
                              cloud_id_in_field,        # 1-D array of unique IDs
                              max_size):                # per-cloud hard cap
    """
    Returns a Dict[int -> int16[:, ::1]]
        key   : cloud ID
        value : 2×N array with the exact #pixels (N ≤ max_size)
                 row coords in axis=0, col coords in axis=1
    Memory use ≈ Σ( N_cloud × 2 × 2 bytes ) with zero over-allocation.
    """

    # -- first pass: collect coordinates in typed.Lists --------------------
    coord_lists = typed.Dict.empty(                     # type: Dict[int, List[(int16,int16)]]
        key_type=types.int64,
        value_type=list_type
    )

    ny, nx = cloudtracknumber_field.shape[1:]

    for row in range(ny):
        for col in range(nx):
            cid = cloudtracknumber_field[0, row, col]
            if cid == 0:
                continue          # background pixel – ignore

            if cid not in coord_lists:
                coord_lists[cid] = typed.List.empty_list(coord_type)

            lst = coord_lists[cid]
            if len(lst) < max_size:              # honour the user-supplied cap
                lst.append((np.int16(row), np.int16(col)))

    # -- second pass: pack each list into a perfectly-sized 2×N array ------
    result = typed.Dict.empty(                     # type: Dict[int, int16[:,::1]]
        key_type=types.int64,
        value_type=array2d_type
    )

    for cid in coord_lists:
        lst = coord_lists[cid]
        n = len(lst)
        arr = np.empty((2, n), dtype=np.int16)

        for i in range(n):
            rc = lst[i]
            arr[0, i] = rc[0]     # row
            arr[1, i] = rc[1]     # col

        result[cid] = arr

    return result


class CoordinateTransformer:
    def __init__(self, target_shape, agg_fact):
        self.agg_fact = agg_fact
        self.target_shape = target_shape

    def transform(self, lat_ind, lon_ind):
        transformed_lat_ind = np.empty(
            (len(lat_ind)*self.agg_fact**2), dtype=int)
        transformed_lon_ind = np.empty(
            (len(lon_ind)*self.agg_fact**2), dtype=int)
        step = self.agg_fact**2
        for k in range(step):
            i = k//self.agg_fact
            j = k % self.agg_fact
            transformed_lat_ind[k::step] = lat_ind*self.agg_fact+i
            transformed_lon_ind[k::step] = lon_ind*self.agg_fact+j
        mask = (transformed_lat_ind < self.target_shape[0]) & (
            transformed_lon_ind < self.target_shape[1])
        # print(mask)
        transformed_lon_ind = transformed_lon_ind[mask]
        transformed_lat_ind = transformed_lat_ind[mask]
        return transformed_lat_ind.T, transformed_lon_ind.T


def extract_value(val):
    if isinstance(val, xr.DataArray):
        return val.values.item() if val.size == 1 else val.values
    return val

# In wgs84

In [18]:


class LatLonCoordinates:
    def __init__(self, lat, lon, is_resampled, agg_fact, pole, temp_key, tracking_fps):
        try:
            with xr.open_dataset(tracking_fps[pole][temp_key]["cloudtracks"][0]) as cloudtrack_data:
                if is_resampled:
                    lat_1d = cloudtrack_data['lat'].values
                    lon_1d = cloudtrack_data['lon'].values
                    print(f"Resampled lat shape: {lat_1d.shape}, lon shape: {lon_1d.shape}")
                    print(lat_1d)
                    self.lat = np.tile(lat_1d[:, np.newaxis], (1, lon_1d.shape[0]))
                    self.lat = np.tile(self.lat[np.newaxis, :, :], (2, 1, 1))
                    self.lon = np.tile(lon_1d[np.newaxis, :], (lat_1d.shape[0], 1))
                    self.lon = np.tile(self.lon[np.newaxis, :, :], (2, 1, 1))
                    print(f"Tiled array shape lat: {self.lat.shape}, lon shape: {self.lon.shape}")
                    self._extract_resampled_coord()
                else:
                    self.lat = lat.values
                    self.lon = lon.values
                    self.coord_transformer = CoordinateTransformer(
                        lon.shape[1:], agg_fact)
        except Exception as e:
            raise RuntimeError(f"Skipping {pole} {temp_key} due to error: {e}")

    def _extract_resampled_coord(self):
        self.lat_resolution = (self.lat.max()-self.lat.min())/len(self.lat)
        self.lon_resolution = (self.lon.max()-self.lon.min())/len(self.lon)


def extract_tracknumbers_data(pole, temp_key, tracking_fps):
    try:
        with xr.open_dataset(tracking_fps[pole][temp_key]["tracknumbers"]) as tracknumbers_data:
            return pd.to_datetime(tracknumbers_data['basetimes'])
    except Exception as e:
        print(f"Skipping {pole} {temp_key} due to error: {e}")
        return None


def extract_trackstats(pole, temp_key, tracking_fps):
    try:
        with xr.open_dataset(tracking_fps[pole][temp_key]["trackstats_final"]) as trackstats_data:
            return trackstats_data.variables['track_duration'].shape[0]
    except Exception as e:
        print(f"Skipping {pole} {temp_key} due to error: {e}")
        return None


def extract_cloud_number_field(cloudtrack_data):
    cloudtracknumber_field = cloudtrack_data['tracknumber'].data
    cloudtracknumber_field[np.isnan(cloudtracknumber_field)] = 0
    return cloudtracknumber_field.astype(int)


def extract_cpp_vars(time, pole, config):
    if time > config["struct_boundary_date"]:
        cpp_filename = time.strftime(
            "%Y/%m/%d/CPPin%Y%m%d%H%M%S405SVMSGI1MD.nc")
    else:
        cpp_filename = time.strftime(
            "%Y/%m/%d/CPPin%Y%m%d%H%M%S405SVMSG01MD.nc")
    # with xr.open_dataset(os.path.join(config["CLAAS_fp"], pole, cpp_filename), chunks="auto") as cpp_data:
    with xr.open_dataset(os.path.join(config["CLAAS_fp"], pole, cpp_filename)) as cpp_data:
        return cpp_data['cot'].values, cpp_data['cwp'].values


def extract_ctx_vars(time, pole, config):
    if time > config["struct_boundary_date"]:
        ctx_filename = time.strftime(
            "%Y/%m/%d/CTXin%Y%m%d%H%M%S405SVMSGI1MD.nc")
    else:
        ctx_filename = time.strftime(
            "%Y/%m/%d/CTXin%Y%m%d%H%M%S405SVMSG01MD.nc")
    # with xr.open_dataset(os.path.join(config["CLAAS_fp"], pole, ctx_filename),chunks="auto") as ctx_data:
    with xr.open_dataset(os.path.join(config["CLAAS_fp"], pole, ctx_filename)) as ctx_data:
        return ctx_data['ctp'].values, ctx_data['ctt'].values

# def extract_dardar_vars(time, config):
#     dardar_filename = time.strftime("%Y/%m/%d/DD_CT_%Y%m%d_%H%M.nc")

#     with xr.open_dataset(os.path.join(config["DARDAR_CPH_fp"], dardar_filename)) as ds:
#         # make them (lat_bin, lon_bin)
#         cph = ds["cph_mean"].isel(time_bin=0).values
#         cth = ds["cth_mean"].isel(time_bin=0).values
#         cth_std = ds["cth_std"].isel(time_bin=0).values
#     return cph, cth, cth_std

def extract_dardar_vars(time, config, lat , lon):
    n_lon, n_lat =  len(lon), len(lat)
    shp = ( n_lat, n_lon)

    def nan_out():
        return (np.full(shp, np.nan, np.float32),
                np.full(shp, np.nan, np.float32),
                np.full(shp, np.nan, np.float32))

    rel = time.strftime("%Y/%m/%d/DD_CT_%Y%m%d_%H%M.nc")
    fp = os.path.join(config["DARDAR_CPH_fp"], rel)
    if not os.path.isfile(fp):
        return nan_out()
    
    with xr.open_dataset(fp) as ds:
        cph = ds["cph_mean"].isel(time_bin=0).values
        cth = ds["cth_mean"].isel(time_bin=0).values
        cth_std = ds["cth_std"].isel(time_bin=0).values
    return cph, cth, cth_std

def extract_dardar_cords(time, config):
    dardar_filename = time.strftime(
            "%Y/%m/%d/DD_CT_%Y%m%d_%H%M.nc")
    with xr.open_dataset(os.path.join(config["DARDAR_CPH_fp"], dardar_filename)) as dardar_data:
        return dardar_data['lat_bin'].values, dardar_data['lon_bin'].values



def extract_aux_vars(aux_ind, cloud_location_ind_non_agg, pix_arr, lat_arr, lon_arr):
    ind1 = cloud_location_ind_non_agg[0]
    ind2 = cloud_location_ind_non_agg[1]
    return pix_arr[aux_ind, ind1, ind2], lat_arr[aux_ind, ind1, ind2], lon_arr[aux_ind, ind1, ind2]


def extract_additional_values(cot_arr, ctp_arr, ctt_arr, cloud_location_ind_non_agg):
    ind1 = cloud_location_ind_non_agg[0]
    ind2 = cloud_location_ind_non_agg[1]
    return cot_arr[0, ind1, ind2], ctp_arr[0, ind1, ind2], ctt_arr[0, ind1, ind2]


def save_single_temp_range_results(cloud_arr, pole, min_temp, max_temp, config):
    columns = ["tracknumber","is_large_pix_cloud", "is_cot_valid_cloud", "is_ctp_valid_cloud", "is_liq", "is_mix", "is_ice", "max_water_frac",
               "max_ice_fraction", "avg_size[km]", "max_size[km]",
               "min_size[km]", "avg_size[px]", "max_size[px]",
               "min_size[px]", "track_start_time", "track_length", "avg_cot", "avg_ctp", "avg_ctt",
               "glaciation_start_time", "glaciation_end_time", "avg_lat",
               "avg_lon", "start_ice_fraction", "end_ice_fraction",
               "ice_frac_hist", "cot_hist", "cot_std_hist",  "cot_nan_frac_hist", "ctp_hist", "ctp_std_hist", "ctp_nan_frac_hist", "ctt_hist", "ctt_std_hist" , "lat_hist", "lon_hist",
               "size_hist_km"]
    if config["validation_mode"] == "dardar":
        columns.extend(["dd_ice_frac_hist", "dd_ice_frac_std_hist","dd_ice_frac_dev_hist", "dd_cth_hist", "dd_cth_std_hist", "dd_pix_claas_if_hist"])
    datapoints_per_cloud = len(columns)
    cloudinfo_df = pd.DataFrame(
        index=range(len(cloud_arr)), columns=columns)
    for cloud_ind in range(len(cloud_arr)):
        current_cloud = cloud_arr[cloud_ind]
        if current_cloud is not None:
            variable_list = [
                current_cloud.id,
                current_cloud.large_pixel_cloud,
                current_cloud.valid_cot_cloud,
                current_cloud.valid_ctp_cloud,
                current_cloud.is_liq,
                current_cloud.is_mix,
                current_cloud.is_ice,
                current_cloud.max_water_fraction,
                current_cloud.max_ice_fraction,
                extract_value(current_cloud.avg_cloud_size_km),
                extract_value(current_cloud.max_size_km),
                extract_value(current_cloud.min_size_km),
                extract_value(current_cloud.avg_cloud_size_px),
                extract_value(current_cloud.max_size_px),
                extract_value(current_cloud.min_size_px),
                current_cloud.track_start_time,
                current_cloud.track_length,
                current_cloud.avg_cot,
                current_cloud.avg_ctp,
                current_cloud.avg_ctt,
                current_cloud.glaciation_start_time,
                current_cloud.glaciation_end_time,
                extract_value(current_cloud.avg_cloud_lat),
                extract_value(current_cloud.avg_cloud_lon),
                current_cloud.start_ice_fraction_arr,
                current_cloud.end_ice_fraction_arr,
                current_cloud.ice_fraction_list,
                current_cloud.mean_cot_list,
                current_cloud.std_cot_list,
                current_cloud.cot_nan_frac_list,
                current_cloud.mean_ctp_list,
                current_cloud.std_ctp_list,
                current_cloud.ctp_nan_frac_list,
                current_cloud.mean_ctt_list,
                current_cloud.std_ctt_list,
                current_cloud.lat_list,
                current_cloud.lon_list,
                current_cloud.cloud_size_km_list
            ]
            if config["validation_mode"] == "dardar":
                variable_list.extend([
                    current_cloud.dd_cph_list,
                    current_cloud.dd_cph_std_list,
                    current_cloud.dd_cph_deviation,
                    current_cloud.dd_cth_list,
                    current_cloud.dd_cth_std_list,
                    current_cloud.dd_pix_claas_cph_meas
                ])
            cloudinfo_df.iloc[cloud_ind] = variable_list


    # Ensure output directory exists
    if config["Resample"]:
        output_dir = os.path.join(
            config['postprocessing_output_dir'], pole,
            config['time_folder_name'],
            f"R_Agg_{config['agg_fact']:02}_T_{abs(round(min_temp)):02}_{abs(round(max_temp)):02}"
        )
    else:
        output_dir = os.path.join(
            config['postprocessing_output_dir'], pole,
            config['time_folder_name'],
            f"Agg_{config['agg_fact']:02}_T_{abs(round(min_temp)):02}_{abs(round(max_temp)):02}"
        )

    os.makedirs(os.path.dirname(output_dir), exist_ok=True)

    # Save DataFrame to Parquet
    output_dir_parq = output_dir + ".parquet"
    print("Writing to ", output_dir_parq)
    cloudinfo_df.to_parquet(output_dir_parq)

    # Optionally save as CSV
    if config['write_csv']:
        output_dir_csv = output_dir + ".csv"
        cloudinfo_df.to_csv(output_dir_csv)

# Latitude and Longitude arrays to pixel area array
def lat_lon_to_pix_arr(lat_arr,lon_arr):
    """
    Convert latitude and longitude arrays to pixel areas based on a Spherical earth model
    
    Parameters:
    lat_arr (array-like): 2D Array of latitude values.
    lon_arr (array-like): 2D Array of longitude values.

    Returns:
    tuple: Two arrays containing the pixel indices for latitude and longitude.
    """
    res_lat = np.pad(abs(lat_arr[0,1:,0] - lat_arr[0,:-1,0]),(0,1), 'edge')  # degrees per pixel latitude direction
    res_lon = np.pad(abs(lon_arr[0,0,1:] - lon_arr[0,0,:-1]),(0,1), 'edge')   # degrees per pixel longitude direction
    return 110*110*np.outer(res_lat,res_lon)[np.newaxis,:,:]*np.cos(np.deg2rad(lat_arr)) 

# @profile
def analyze_single_temp_range(temp_ind: int, tracking_fps: dict, pole: str, config: dict, pix_area: np.array = None, pix_area_agg: np.array  = None,  lon: np.array =None, lat: np.array =None, lat_agg: np.array  = None, lon_agg: np.array =None) -> None:
    """
    Analises a given cloud top temperature range and saves the results in a dataframe.
    
    Parameters:
        temp_ind (int): Index of the temperature range to analyze.
        tracking_fps (dict): Dictionary containing file paths for the tracking data (PyFlexTrkr output).
        pole (str): Pole to analyze ("N" or "S", also known as hemisphere :)).
        config (dict): Configuration dictionary containing parameters for the analysis. Those are the gte_config.yaml conntents .
        pix_area (array-like, optional): 3D (x,y,1) Array of pixel area values for non-resampled data. Required if config["Resample"] is False.
        pix_area_agg (array-like, optional): 3D (x,y,1) Array of pixel area values on the aggregated grid. Required if config["Resample"] is False.
        lon (array-like, optional): 3D (x,y,1) Array of cell longitude values. Required if config["Resample"] is False.
        lat (array-like, optional): 3D (x,y,1) Array of cell latitude values. Required if config["Resample"] is False.
    Returns:
        Nothing
    """
    # Load configuration parameters
    min_temp, max_temp = config['min_temp_arr'][temp_ind], config['max_temp_arr'][temp_ind]
    abs_min_temp, abs_max_temp = abs(round(min_temp)), abs(round(max_temp))
    is_resampled = config["Resample"]
    collect_add_properties = config["collect_additional_properties"]
    temp_key = f'{abs_min_temp}_{abs_max_temp}'
    validation_mode = config['validation_mode']

    # Load datasets
    try:
        cords = LatLonCoordinates(
            lat, lon, is_resampled, config['agg_fact'], pole, temp_key, tracking_fps)
        lat_arr = cords.lat if cords.lat is not None else None
        lon_arr = cords.lon if cords.lon is not None else None
        assert ((lat_arr is None) or (len(lat_arr.shape) == 3)), f"Latitude array is not 3-D, it is {len(lat_arr.shape)}D"
        assert ((lon_arr is None) or (len(lon_arr.shape) == 3)), f"Longitude array is not 3-D, it is {len(lon_arr.shape)}D"
    except Exception as e:
        print(f"Exception in coordinate creation: {e}")
        return None
    basetimes = extract_tracknumbers_data(pole, temp_key, tracking_fps)
    n_tracks = extract_trackstats(pole, temp_key, tracking_fps)
    if basetimes is None or n_tracks is None:
        return None

    print(f"Analyzing {pole} {temp_key} with {n_tracks} tracks")

    cloud_arr = np.empty((n_tracks), dtype=Cloud)
    for i in range(n_tracks):
        cloud_arr[i] = None

    # Cloud(f'{temp_ind}_{i}') for i in range(n_tracks)])
    # print(f"Analyzing T: {min_temp} to {max_temp} Agg={config['agg_fact']}")

    pix_arr = pix_area.values if pix_area is not None else lat_lon_to_pix_arr(lat_arr, lon_arr)
    pix_arr_agg = pix_area_agg.values if pix_area_agg is not None else lat_lon_to_pix_arr(lat_arr, lon_arr)
    
    if validation_mode == "dardar":
        dd_lat , dd_lon = None, None
        lat_arr_agg = lat_agg.values if lat_agg is not None else None
        lon_arr_agg = lon_agg.values if lon_agg is not None else None
    for fp_ind in range(len(basetimes)):
        time = basetimes[fp_ind]
        time_str = time.strftime("%Y%m%d_%H%M%S")
        # print(f'{min_temp} to {max_temp} Loading {time_str}')

        aux_ind = 1 if time > config["struct_boundary_date"] else 0

        if collect_add_properties:
            cot_arr, cwp_arr = extract_cpp_vars(time, pole, config)
            ctp_arr, ctt_arr = extract_ctx_vars(time, pole, config)

        if validation_mode == "dardar":
            if dd_lon is None or dd_lat is None:
                dd_lat, dd_lon = extract_dardar_cords(time, config)
                dardar_index = build_dardar_index(dd_lat, dd_lon)
            dd_cph, dd_cth, dd_cth_std = extract_dardar_vars(time, config, dd_lat, dd_lon)


        cloudtrack_fp = tracking_fps[pole][temp_key]['cloudtracks'][fp_ind]

        with xr.open_dataset(cloudtrack_fp) as cloudtrack_data:
            cph_arr = cloudtrack_data['cph_filtered'].values
            cloudtracknumber_field = extract_cloud_number_field(
                cloudtrack_data)
            cloud_id_in_field, counts = np.unique(
                cloudtracknumber_field, return_counts=True)
            counts = counts[cloud_id_in_field != 0]
            if len(counts) == 0:
                print(
                    f"{pole} - {min_temp} to {max_temp}: No cloud timestep: {time_str}")
                continue
            cloud_id_in_field = cloud_id_in_field[cloud_id_in_field != 0]
            max_allowed_cloud_size_px = config['fast_mode_arr_size'] if config['postprocessing_fast_mode'] else counts.max(
            )
            hash_map_cloud_numbers = extract_cloud_coordinates(
                cloudtracknumber_field, cloud_id_in_field, max_allowed_cloud_size_px)  # counts.max())
            del cloudtracknumber_field

        # print(f"N_clouds in frame {len(cloud_id_in_field)}", flush=True)
        # if max_allowed_cloud_size_px > 1000000:
        #     print(np.where(counts, counts == counts.max()))
        # print(cloud_id_in_field)

        for track_number in cloud_id_in_field:

            try:
                if cloud_arr[track_number-1] is None:
                    cloud_arr[track_number-1] = Cloud(track_number, is_resampled)
            except:
                print(
                    f"Error: {temp_ind,track_number,len(cloud_arr)}")
                continue

            if (not cloud_arr[track_number-1].terminate_cloud):
                # TODO:SPEED UP NEXT TWO LINES (set_cloud_values and update_status)
                cord = hash_map_cloud_numbers[track_number]
                cloud_location_ind = [cord[0, :], cord[1, :]]

                if cloud_location_ind[0].size != 0:
                    cloud_cph_values = cph_arr[0,
                                               cloud_location_ind[0].T, cloud_location_ind[1].T]
                    # print(f"Cloud cph values size: {cloud_cph_values.shape}")
                    # print(f"Cloud loc ind 0 size: {cloud_location_ind[0].shape}")
                    # print(f"Cloud loc ind 1 size: {cloud_location_ind[1].shape}")
                    if is_resampled:
                        # avg_lat_ind = int(
                        #     round(np.mean(cloud_location_ind[0])))
                        # avg_lon_ind = int(
                        #     round(np.mean(cloud_location_ind[1])))
                        cloud_pix_area_values, cloud_lat_values, cloud_lon_values = extract_aux_vars(
                            aux_ind, cloud_location_ind, pix_arr, lat_arr, lon_arr)
                        agg_pix_area_values = pix_arr_agg[aux_ind, cloud_location_ind[0].T, cloud_location_ind[1].T]
                        # TODO:SPEED UP NEXT TWO LINES (set_cloud_values and update_status)
                        # cloud_arr[track_number-1].update_status(
                        #     time, cloud_cph_values, extract_value(cords.lat[avg_lat_ind]), extract_value(cords.lon[avg_lon_ind]), pixel_area=cords.lat_resolution.values*cords.lon_resolution.values)
                        if collect_add_properties:
                            cloud_cot_values, cloud_ctp_values, cloud_ctt_values = extract_additional_values(
                                cot_arr, ctp_arr, ctt_arr, cloud_location_ind)
                        else:
                            cloud_cot_values, cloud_ctp_values, cloud_ctt_values = np.array(
                                []), np.array([]), np.array([])
                        cloud_arr[track_number-1].update_status(
                            time, cloud_cph_values, cloud_cot_values, cloud_ctp_values, cloud_ctt_values, cloud_lat_values, cloud_lon_values, cloud_pix_area_values, agg_pix_area_values)

                    else:
                        cloud_location_ind_non_agg = cords.coord_transformer.transform(
                            cloud_location_ind[0], cloud_location_ind[1])
                        cloud_pix_area_values, cloud_lat_values, cloud_lon_values = extract_aux_vars(
                            aux_ind, cloud_location_ind_non_agg, pix_arr, lat_arr, lon_arr)
                        agg_pix_area_values = pix_arr_agg[aux_ind, cloud_location_ind[0].T, cloud_location_ind[1].T]
                        if not (agg_pix_area_values > 0).all():
                            print(f"0 or negative values in aggregated pixel area array {agg_pix_area_values}")
                            print(len(agg_pix_area_values))
                            print(pix_arr_agg.shape)
                            print(cloud_location_ind)
                        # print(f"Cloud location ind non agg 0: {cloud_location_ind_non_agg[0].shape}")
                        # print(f"Cloud pix area values: {cloud_pix_area_values[::9].shape}")
                        # assert (cloud_pix_area_values[::9].shape == cloud_cph_values.shape), f"Pixel area size array mismatch\npix_area:{cloud_pix_area_values[::9].shape}\n{cloud_cph_values.shape}\ncloud_location_ind 0: {cloud_location_ind[0]}\ncloud_location_ind 1: {cloud_location_ind[1]}\ncloud_location_ind_non_agg 0: {cloud_location_ind_non_agg[0][:-20]}\ncloud_location_ind_non_agg 1: {cloud_location_ind_non_agg[1][:-20]}"
                        if collect_add_properties:
                            cloud_cot_values, cloud_ctp_values, cloud_ctt_values = extract_additional_values(
                                cot_arr, ctp_arr, ctt_arr, cloud_location_ind_non_agg)
                        else:
                            cloud_cot_values, cloud_ctp_values, cloud_ctt_values = np.array(
                                []), np.array([]), np.array([])
                        # print(np.info(cloud_cot_values))
                        # assert (cloud_pix_area_values.size >= ((cloud_cph_values.size-1) * (config['agg_fact'] ** 2))),  "Pixel area size array mismatch"
                        cloud_dd_cph=None
                        cloud_dd_cth=None
                        cloud_dd_cth_std=None
                        if validation_mode == "dardar":
                            # Match DARDAR to EACH cloud pixel location
                            agg_lat_values = lat_arr_agg[aux_ind, cloud_location_ind[0].T, cloud_location_ind[1].T]
                            agg_lon_values = lon_arr_agg[aux_ind, cloud_location_ind[0].T, cloud_location_ind[1].T]
                            cloud_dd_cph, cloud_dd_cth, cloud_dd_cth_std = match_dardar_to_cloud(
                                dardar_index,
                                dd_cph, dd_cth, dd_cth_std,
                                agg_lat_values, agg_lon_values,
                                max_km=config.get("dardar_max_match_km", None),  # optional
                                fill_value=np.nan
                                )
                        cloud_arr[track_number-1].update_status(
                            time,
                            cloud_cph_values, cloud_cot_values, cloud_ctp_values, cloud_ctt_values,
                            cloud_lat_values, cloud_lon_values, cloud_pix_area_values, agg_pix_area_values,
                            dd_cph=cloud_dd_cph, dd_cth=cloud_dd_cth, dd_cth_std=cloud_dd_cth_std)

                else:
                    cloud_arr[track_number-1].update_missing_cloud()
        if collect_add_properties:
            del ctp_arr, cwp_arr, cot_arr, ctt_arr
        del cph_arr
        del cloud_cot_values, cloud_ctp_values, cloud_cph_values, cloud_ctt_values
        del hash_map_cloud_numbers
        if not is_resampled:
            del cloud_location_ind_non_agg
        del cloud_location_ind
        del cloud_pix_area_values, cloud_lat_values, cloud_lon_values
 
    save_single_temp_range_results(cloud_arr, pole, min_temp, max_temp, config)



def analize_single_pole(pole, cloud_dict, tracking_fps, config):
    print(f"Analyzing {pole}")
    aux_ds = xr.load_dataset(config["aux_fps"][pole], decode_times=False)
    aux_ds_agg = xr.load_dataset(config["aux_fps_agg"][pole], decode_times=False)
    n_procs = config.get("n_postproc_cores",4)
    if config["Resample"]:
        with Pool(n_procs) as pool:
            part_single_temp_range = partial(
                analyze_single_temp_range, tracking_fps=tracking_fps, pole=pole, config=config)
            pool.map(part_single_temp_range, range(
                len(config['min_temp_arr'])))
            pool.close()
            pool.join()
    if not config["Resample"]:
        lat_mat = aux_ds["lat"].load()
        lon_mat = aux_ds["lon"].load()
        pix_area = aux_ds["pixel_area"].load()
        pix_area_agg = aux_ds_agg["pixel_area"].load()
        lat_agg = aux_ds_agg["lat"].load()
        lon_agg = aux_ds_agg["lon"].load()
        assert (~np.isnan(pix_area_agg.values).any()), "NaN values in aggregated pixel area array"
        assert (~np.isnan(lat_agg.values).any()), "NaN values in aggregated pixel area array"
        if config.get("validation_mode", None) == "dardar":
            part_single_temp_range = partial(analyze_single_temp_range, tracking_fps=tracking_fps,
                                            pole=pole, config=config, pix_area=pix_area, pix_area_agg = pix_area_agg, lon=lon_mat, lat=lat_mat, lat_agg = lat_agg, lon_agg = lon_agg)
        else: 
            part_single_temp_range = partial(analyze_single_temp_range, tracking_fps=tracking_fps,
                                            pole=pole, config=config, pix_area=pix_area, pix_area_agg = pix_area_agg, lon=lon_mat, lat=lat_mat)
        with Pool(n_procs) as pool:
            pool.map(part_single_temp_range, range(
                len(config['min_temp_arr'])))
            pool.close()
            pool.join()
        # for ind in range(len(config['min_temp_arr'])):
        #     part_single_temp_range(ind)



def analyze_tracked_clouds(config):
    tracking_fps=generate_tracking_filenames(config)
    with Manager() as manager:
        cloud_dict=manager.dict()
        # TODO: Paralelize here
        part_analize_single_pole=partial(
            analize_single_pole, cloud_dict=cloud_dict, tracking_fps=tracking_fps, config=config)
        # with NestablePool(2) as pool:
        #     pool.map(part_analize_single_pole, config['pole_folders'])
        #     pool.close()
        #     pool.join()
        for pole in config['pole_folders']:
            part_analize_single_pole(pole)


In [19]:
config = read_config("/cluster/work/climate/dnikolo/n2o/Glaciation_time_estimator/configs/Validation/valid_01_10_may_2007.yaml")
tracking_fps = generate_tracking_filenames(config)
with Manager() as manager:
    cloud_dict = manager.dict()
    # TODO: Paralelize here
    part_analize_single_pole = partial(
        analize_single_pole, cloud_dict=cloud_dict, tracking_fps=tracking_fps, config=config)
    part_analize_single_pole("np")
    # with NestablePool(2) as pool:
    #     pool.map(part_analize_single_pole, config['pole_folders'])
    #     pool.close()

# def analyse_tracked_clouds(config):
#     tracking_fps = generate_tracking_filenames(config)
#     with Manager() as manager:
#         cloud_dict = manager.dict()
#         # TODO: Paralelize here
#         part_analize_single_pole = partial(
#             analize_single_pole, cloud_dict=cloud_dict, tracking_fps=tracking_fps, config=config)
#         with NestablePool(2) as pool:
#             pool.map(part_analize_single_pole, config['pole_folders'])
#             pool.close()
#             pool.join()

Analyzing np
Analyzing np 6_0 with 10231 tracksAnalyzing np 12_6 with 14348 tracks

Analyzing np 18_12 with 14164 tracks
Analyzing np 24_18 with 10206 tracks
Checking dardar cloud
Checking dardar cloud
Checking dardar cloud
Checking dardar cloud
Checking dardar cloud
Checking dardar cloud
Checking dardar cloud
Checking dardar cloud
Checking dardar cloud
Checking dardar cloud
Checking dardar cloud
Checking dardar cloud
Checking dardar cloud
Checking dardar cloud
Checking dardar cloud
Checking dardar cloud
Checking dardar cloud
Checking dardar cloud
Checking dardar cloud
Checking dardar cloud
Checking dardar cloud
Checking dardar cloud
Checking dardar cloud
Checking dardar cloud
Checking dardar cloud
Checking dardar cloud
Checking dardar cloud
Checking dardar cloud
Checking dardar cloud
Checking dardar cloud
Checking dardar cloud
Checking dardar cloud
Checking dardar cloud
Checking dardar cloud
Checking dardar cloud
Checking dardar cloud
Checking dardar cloud
Checking dardar cloud
Checki

KeyboardInterrupt: 

## Old version

In [11]:



def extract_value(val):
    if isinstance(val, xr.DataArray):
        return val.values.item() if val.size == 1 else val.values
    return val


def extract_cpp_vars(time, pole):
    cpp_filename = time.strftime("CPPin%Y%m%d%H%M%S405SVMSGI1MD.nc")
    with xr.load_dataset(os.path.join(os.environ["TMPDIR"], "Data", pole, cpp_filename)) as cpp_data:
        return cpp_data['cot'],cpp_data['cwp']

def extract_ctx_vars(time, pole):
    ctx_filename = time.strftime("CTXin%Y%m%d%H%M%S405SVMSGI1MD.nc")
    with xr.load_dataset(os.path.join(os.environ["TMPDIR"], "Data", pole, ctx_filename)) as ctx_data:
        return ctx_data['ctp']
    # print(f'{min_temp} to {max_temp} Loading {time_str}')
# /cluster/work/climate/dnikolo/dump/Data/np/CPPin20210101000000405SVMSGI1MD.nc


def extract_cloud_number_field(cloudtrack_data):
    cloudtracknumber_field = cloudtrack_data['tracknumber'].data
    cloudtracknumber_field[np.isnan(cloudtracknumber_field)] = 0
    return cloudtracknumber_field.astype(int)


def save_single_temp_range_results(cloud_arr, pole, min_temp, max_temp, config):
    columns = ["is_large_pix_cloud", "is_cot_valid_cloud","is_ctp_valid_cloud", "is_liq", "is_mix", "is_ice", "max_water_frac",
               "max_ice_fraction", "avg_size[km]", "max_size[km]",
               "min_size[km]", "avg_size[px]", "max_size[px]",
               "min_size[px]", "track_start_time", "track_length", "avg_cot","avg_ctp",
               "glaciation_start_time", "glaciation_end_time", "avg_lat",
               "avg_lon", "start_ice_fraction", "end_ice_fraction",
               "ice_frac_hist", "cot_hist", "cot_nan_frac_hist","ctp_hist", "ctp_nan_frac_hist", "lat_hist", "lon_hist",
               "size_hist_km"]
    datapoints_per_cloud = len(columns)
    cloudinfo_df = pd.DataFrame(
        index=range(len(cloud_arr)), columns=columns)
    for cloud_ind in range(len(cloud_arr)):
        current_cloud = cloud_arr[cloud_ind]
        if current_cloud is not None:
            cloudinfo_df.iloc[cloud_ind] = [
                current_cloud.large_pixel_cloud,
                current_cloud.valid_cot_cloud,
                current_cloud.valid_ctp_cloud,
                current_cloud.is_liq,
                current_cloud.is_mix,
                current_cloud.is_ice,
                current_cloud.max_water_fraction,
                current_cloud.max_ice_fraction,
                extract_value(current_cloud.avg_cloud_size_km),
                extract_value(current_cloud.max_size_km),
                extract_value(current_cloud.min_size_km),
                extract_value(current_cloud.avg_cloud_size_px),
                extract_value(current_cloud.max_size_px),
                extract_value(current_cloud.min_size_px),
                current_cloud.track_start_time,
                current_cloud.track_length,
                current_cloud.avg_cot,
                current_cloud.avg_ctp,
                current_cloud.glaciation_start_time,
                current_cloud.glaciation_end_time,
                extract_value(current_cloud.avg_cloud_lat),
                extract_value(current_cloud.avg_cloud_lon),
                current_cloud.start_ice_fraction_arr,
                current_cloud.end_ice_fraction_arr,
                current_cloud.ice_fraction_list,
                current_cloud.mean_cot_list,
                current_cloud.cot_nan_frac_list,
                current_cloud.mean_ctp_list,
                current_cloud.ctp_nan_frac_list,
                current_cloud.lat_list,
                current_cloud.lon_list,
                current_cloud.cloud_size_km_list
            ]

    # Ensure output directory exists
    output_dir = os.path.join(
        config['postprocessing_output_dir'], pole,
        config['time_folder_name'],
        f"Agg_{config['agg_fact']:02}_T_{abs(round(min_temp)):02}_{abs(round(max_temp)):02}"
    )
    os.makedirs(os.path.dirname(output_dir), exist_ok=True)

    # Save DataFrame to Parquet
    output_dir_parq = output_dir + ".parquet"
    print("Writing to ", output_dir_parq)
    cloudinfo_df.to_parquet(output_dir_parq)

    # Optionally save as CSV
    if config['write_csv']:
        output_dir_csv = output_dir + ".csv"
        cloudinfo_df.to_csv(output_dir_csv)


def analize_single_temp_range(temp_ind: int, cloud_dict, tracking_fps: dict, pole: str, config: dict, pix_area=None,  lon=None, lat=None) -> None:
    # loop_start_time=dt.datetime.now()
    min_temp, max_temp = config['min_temp_arr'][temp_ind], config['max_temp_arr'][temp_ind]
    is_resampled = config["Resample"]
    collect_cot = config["collect_additional_properties"]
    # Load datasets
    temp_key = f'{abs(round(min_temp))}_{abs(round(max_temp))}'
    print(f"Analyzing {pole} {temp_key}")
    # print(tracking_fps[pole][temp_key]["cloudtracks"][0])
    # print(tracking_fps[pole][temp_key]["trackstats_final"])
    # print(tracking_fps[pole][temp_key]["tracknumbers"])
    try:
        # print(tracking_fps[pole][temp_key]["cloudtracks"][0])
        cloudtrack_data = xr.load_dataset(
            tracking_fps[pole][temp_key]["cloudtracks"][0])
        trackstats_data = xr.load_dataset(
            tracking_fps[pole][temp_key]["trackstats_final"])
        tracknumbers_data = xr.load_dataset(
            tracking_fps[pole][temp_key]["tracknumbers"])
    except:  # Exception as inst:
        print(f"Skipping {pole} {min_temp} to {max_temp}")
        cloud_dict[temp_key] = np.array([])
        return None
    # Load relevant data from datasets into local variables
    n_tracks = trackstats_data.variables['track_duration'].shape[0]
    basetimes = pd.to_datetime(tracknumbers_data['basetimes'])
    if is_resampled:
        lat = cloudtrack_data['lat']
        lon = cloudtrack_data['lon']
        lat_resolution = (lat.max()-lat.min())/len(lat)
        lon_resolution = (lon.max()-lon.min())/len(lon)
    else:
        coord_transformer = CoordinateTransformer(
            lon.shape[1:], config["agg_fact"])
    trackstats_data.close()
    tracknumbers_data.close()
    cloudtrack_data.close()
    # print(append_start_time-loop_start_time)
    cloud_arr = np.empty((n_tracks), dtype=Cloud)
    # Cloud(f'{temp_ind}_{i}') for i in range(n_tracks)])
    # print(append_end_time-append_start_time)
    # print(f"Analyzing T: {min_temp} to {max_temp} Agg={config['agg_fact']}")
    for fp_ind in range(len(basetimes)):
        time = basetimes[fp_ind]
        time_str = time.strftime("%Y%m%d_%H%M%S")
        print(f'{min_temp} to {max_temp} Loading {time_str}')
        if collect_cot:
            cot_field,cwp_field = extract_cpp_vars(time, pole)
            ctp_field  = extract_ctx_vars(time, pole)
        cloudtrack_fp = tracking_fps[pole][temp_key]['cloudtracks'][fp_ind]
        cloudtrack_data = xr.load_dataset(cloudtrack_fp)
        cloudtracknumber_field = extract_cloud_number_field(cloudtrack_data)
        cph_field = cloudtrack_data['cph_filtered']
        cloud_id_in_field, counts = np.unique(
            cloudtracknumber_field, return_counts=True)
        counts = counts[cloud_id_in_field != 0]
        if len(counts) == 0:
            continue
        cloud_id_in_field = cloud_id_in_field[cloud_id_in_field != 0]
        max_allowed_cloud_size_px = config['fast_mode_arr_size'] if config['postprocessing_fast_mode'] else counts.max(
        )
        hash_map_cloud_numbers = extract_cloud_coordinates(
            cloudtracknumber_field, cloud_id_in_field, max_allowed_cloud_size_px)  # counts.max())
        cloudtrack_data.close()
        if max_allowed_cloud_size_px > 1000000:
            print(np.where(counts, counts == counts.max()))
        # print(cloud_id_in_field)
        for track_number in cloud_id_in_field:
            try:
                if cloud_arr[track_number-1] is None:
                    cloud_arr[track_number-1] = Cloud(temp_key, is_resampled)
            except:
                print(
                    f"Error: {temp_ind,track_number,len(cloud_arr)}")
                continue

            if (not cloud_arr[track_number-1].terminate_cloud):
                # TODO:SPEED UP NEXT TWO LINES (set_cloud_values and update_status)
                ind, cord = hash_map_cloud_numbers[track_number]
                cloud_location_ind = [cord[0, :ind], cord[1, :ind]]
                if cloud_location_ind[0].size != 0:
                    cloud_cph_values = cph_field.values[0,
                                                        cloud_location_ind[0].T, cloud_location_ind[1].T]
                    if is_resampled:
                        avg_lat_ind = int(
                            round(np.mean(cloud_location_ind[0])))
                        avg_lon_ind = int(
                            round(np.mean(cloud_location_ind[1])))
                        # TODO:SPEED UP NEXT TWO LINES (set_cloud_values and update_status)
                        cloud_arr[track_number-1].update_status(
                            time, cloud_cph_values, extract_value(lat[avg_lat_ind]), extract_value(lon[avg_lon_ind]), pixel_area=lat_resolution.values*lon_resolution.values)
                    else:
                        cloud_location_ind_non_agg = coord_transformer.transform(
                            cloud_location_ind[0], cloud_location_ind[1])
                        cloud_cph_values = cph_field.values[0,
                                                            cloud_location_ind[0].T, cloud_location_ind[1].T]
                        cloud_pix_area_values = pix_area.values[0,
                                                                cloud_location_ind_non_agg[0], cloud_location_ind_non_agg[1]]
                        cloud_lat_values = lat.values[0,
                                                      cloud_location_ind_non_agg[0], cloud_location_ind_non_agg[1]]
                        cloud_lon_values = lon.values[0,
                                                      cloud_location_ind_non_agg[0], cloud_location_ind_non_agg[1]]
                        if collect_cot:
                            cloud_cot_values = cot_field.values[0,
                                                                cloud_location_ind_non_agg[0], cloud_location_ind_non_agg[1]]
                            cloud_ctp_values = ctp_field.values[0,
                                                                cloud_location_ind_non_agg[0], cloud_location_ind_non_agg[1]]
                        else:
                            cloud_cot_values = snp.array([0])
                            cloud_ctp_values = np.array([0])
                        # print(np.info(cloud_cot_values))
                        cloud_arr[track_number-1].update_status(
                            time, cloud_cph_values, cloud_cot_values, cloud_ctp_values, cloud_lat_values, cloud_lon_values, cloud_pix_area_values)
                        
                else:
                    cloud_arr[track_number-1].update_missing_cloud()
    save_single_temp_range_results(cloud_arr, pole, min_temp, max_temp, config)



def analize_single_pole(pole, cloud_dict, tracking_fps, config):
    print(f"Analyzing {pole}")
    aux_ds = xr.load_dataset(config["aux_fps"][pole], decode_times=False)
    aux_ds_agg = xr.load_dataset(config["aux_fps_agg"][pole], decode_times=False)
    n_procs = config.get("n_postproc_cores",4)
    if config["Resample"]:
        with Pool(n_procs) as pool:
            part_single_temp_range = partial(
                analyze_single_temp_range, tracking_fps=tracking_fps, pole=pole, config=config)
            pool.map(part_single_temp_range, range(
                len(config['min_temp_arr'])))
            pool.close()
            pool.join()
    if not config["Resample"]:
        lat_mat = aux_ds["lat"].load()
        lon_mat = aux_ds["lon"].load()
        pix_area = aux_ds["pixel_area"].load()
        pix_area_agg = aux_ds_agg["pixel_area"].load()
        lat_agg = aux_ds_agg["lat"].load()
        lon_agg = aux_ds_agg["lon"].load()
        assert (~np.isnan(pix_area_agg.values).any()), "NaN values in aggregated pixel area array"
        assert (~np.isnan(lat_agg.values).any()), "NaN values in aggregated pixel area array"
        if config.get("validation_mode", None) == "dardar":
            part_single_temp_range = partial(analyze_single_temp_range, tracking_fps=tracking_fps,
                                            pole=pole, config=config, pix_area=pix_area, pix_area_agg = pix_area_agg, lon=lon_mat, lat=lat_mat, lat_agg = lat_agg, lon_agg = lon_agg)
        else: 
            part_single_temp_range = partial(analyze_single_temp_range, tracking_fps=tracking_fps,
                                            pole=pole, config=config, pix_area=pix_area, pix_area_agg = pix_area_agg, lon=lon_mat, lat=lat_mat)
        with Pool(n_procs) as pool:
            pool.map(part_single_temp_range, range(
                len(config['min_temp_arr'])))
            pool.close()
            pool.join()
        # for ind in range(len(config['min_temp_arr'])):
        #     part_single_temp_range(ind)



def save_results(res_dict, config):
    min_temp, max_temp = config['min_temp_arr'][0], config['max_temp_arr'][0]
    temp_key = f'{abs(round(min_temp))}_{abs(round(max_temp))}'
    # cloudtrack_data = xr.(
    #     tracking_fps['np'][temp_key]["cloudtracks"][0])
    # lat = cloudtrack_data['lat']
    # lon = cloudtrack_data['lon']
    # lat_resolution = extract_value((lat.max()-lat.min())/len(lat))
    # lon_resolution = extract_value((lon.max()-lon.min())/len(lon))
    # cloudtrack_data.close()
    columns = ["is_liq", "is_mix", "is_ice", "max_water_frac",
               "max_ice_fraction", "avg_size[km]", "max_size[km]",
               "min_size[km]", "avg_size[px]", "max_size[px]",
               "min_size[px]", "track_start_time", "track_length",
               "glaciation_start_time", "glaciation_end_time", "avg_lat",
               "avg_lon", "start_ice_fraction", "end_ice_fraction",
               "ice_frac_hist", "cot_hist", "lat_hist", "lon_hist",
               "size_hist_km"]
    datapoints_per_cloud = len(columns)
    # Iterating through the cloud data
    for temp_ind in range(len(config['max_temp_arr'])):
        for pole in config['pole_folders']:
            min_temp, max_temp = config['min_temp_arr'][temp_ind], config['max_temp_arr'][temp_ind]
            temp_key = f'{abs(round(min_temp))}_{abs(round(max_temp))}'
            key = f'{pole}_{temp_key}'
            cloud_arr = res_dict[key]

            cloudinfo_df = pd.DataFrame(
                index=range(len(cloud_arr)), columns=columns)
            for cloud_ind in range(len(cloud_arr)):
                current_cloud = cloud_arr[cloud_ind]
                if current_cloud is not None:
                    cloudinfo_df.iloc[cloud_ind] = [
                        current_cloud.
                        current_cloud.is_liq,
                        current_cloud.is_mix,
                        current_cloud.is_ice,
                        current_cloud.max_water_fraction,
                        current_cloud.max_ice_fraction,
                        extract_value(current_cloud.avg_cloud_size_km),
                        extract_value(current_cloud.max_size_km),
                        extract_value(current_cloud.min_size_km),
                        extract_value(current_cloud.avg_cloud_size_px),
                        extract_value(current_cloud.max_size_px),
                        extract_value(current_cloud.min_size_px),
                        current_cloud.track_start_time,
                        current_cloud.track_length,
                        current_cloud.glaciation_start_time,
                        current_cloud.glaciation_end_time,
                        extract_value(current_cloud.avg_cloud_lat),
                        extract_value(current_cloud.avg_cloud_lon),
                        current_cloud.start_ice_fraction_arr,
                        current_cloud.end_ice_fraction_arr,
                        current_cloud.ice_fraction_list,
                        current_cloud.mean_cot_list,
                        current_cloud.lat_list,
                        current_cloud.lon_list,
                        current_cloud.cloud_size_km_list
                    ]

            # Ensure output directory exists
            output_dir = os.path.join(
                config['postprocessing_output_dir'],
                config['time_folder_name'],
                f"T_{abs(round(min_temp)):02}_{abs(round(max_temp)):02}_agg_{config['agg_fact']:02}"
            )
            os.makedirs(os.path.dirname(output_dir), exist_ok=True)

            # Save DataFrame to Parquet
            output_dir_parq = output_dir + ".parquet"
            print("Writing to ", output_dir_parq)
            cloudinfo_df.to_parquet(output_dir_parq)

            # Optionally save as CSV
            if config['write_csv']:
                output_dir_csv = output_dir + ".csv"
                cloudinfo_df.to_csv(output_dir_csv)


In [12]:
import numpy as np
import xarray as xr
import datetime as dt



class Cloud:
    # def __new__(self, *args, **kwargs):
    #     return super().__new__(self)
    def __init__(self, cloud_id, is_resampled):
        self.id = cloud_id
        self.is_resampled = is_resampled
        self.crit_fraction = 0.1
        # Bools inidicating if the cloud has been liquid at any point
        self.is_liq: bool = False
        self.is_mix: bool = False
        self.is_ice: bool = False
        # Max and min cloud size in pixels
        self.max_size_km: float = 0.0
        self.max_size_px: int = 0
        self.min_size_km: float = 510.0e6
        self.min_size_px: int = 3717*3717

        # Variables giving the first and last 4 timesteps (1 hour) of the cloud ice fraction - both arrays run in the same time direction start: [1 , 2 , 3 , 4] ... end: [1 , 2 , 3 , 4]
        self.start_ice_fraction_arr = np.empty(4)
        self.end_ice_fraction_arr = np.empty(4)
        # self.ice_fraction_arr=np.empty(max_timesteps)
        self.ice_fraction_list = []
        

        self.max_water_fraction: float = 0.0
        self.max_ice_fraction: float = 0.0

        self.track_start_time: dt.datetime = None
        self.track_end_time: dt.datetime = None
        self.track_length = None

        self.glaciation_start_time: dt.datetime = None
        self.glaciation_end_time: dt.datetime = None

        self.n_timesteps = None

        self.sum_cloud_cot=0
        self.avg_cot = None
        self.cot_timestep_counter=0
        self.mean_cot_list = []
        self.std_cot_list = []

        self.sum_cloud_ctp=0
        self.avg_ctp = None
        self.ctp_timestep_counter=0
        self.mean_ctp_list = []
        self.std_ctp_list = []

        self.sum_cloud_cwp=0
        self.avg_cwp = None
        self.cwp_timestep_counter=0
        self.mean_cwp_list = []
        self.std_cwp_list = []
        
        self.sum_cloud_lat = 0.0
        self.sum_cloud_lon = 0.0
        self.avg_cloud_lat = None
        self.avg_cloud_lon = None
        self.lon_list=[]
        self.lat_list=[]

        self.sum_cloud_size_km = 0.0
        self.avg_cloud_size_km = None
        self.cloud_size_km_list = []
        self.large_pixel_cloud=False

        self.sum_cloud_size_px = 0.0
        self.avg_cloud_size_px = None

        
        self.valid_cot_cloud = False
        self.cot_nan_frac_list=[]

        self.valid_ctp_cloud = False
        self.ctp_nan_frac_list=[]

        self.n_timesteps_no_cloud = 0
        self.terminate_cloud = False

    def __str__(self):
        return f"{self.is_liq},{self.is_mix},{self.is_ice},"
    #In resampled clouds pixel area should be the area in degrees lon_resolution*lat_resolution
    def update_status(self, time: dt.datetime, cloud_values: np.array, cot_values, ctp_values, cloud_lat, cloud_lon ,pixel_area):
        ind_to_take = ~np.isnan(pixel_area)
        pixel_area = pixel_area[ind_to_take]
        cot_values = cot_values[ind_to_take]
        ctp_values = ctp_values[ind_to_take]
        cloud_lat = cloud_lat[ind_to_take]
        cloud_lon = cloud_lon[ind_to_take]
        cloud_size_px = cloud_values.shape[0]
        if not self.is_resampled:
            cloud_lat = np.average(cloud_lat,weights=pixel_area)
            cloud_lon = np.average(cloud_lon,weights=pixel_area)
            # cloud_lat = 10
            # cloud_lon = 10
        # print(cloud_values)
        if cloud_size_px:
            self.n_timesteps_no_cloud = 0
            valid_values = cloud_values[cloud_values >= 1]
            # print(len(valid_values)/len(cloud_values))
            ice_fraction = (valid_values.sum() -
                            float(len(valid_values)))/float(len(valid_values))
            # print(valid_values)
            # ice_fraction=float(np.count_nonzero(cloud_values==2))/float(cloud_size_px)
            water_fraction = 1-ice_fraction
            # assert math.isclose(water_fraction+ice_fraction,1)
            # print(water_fraction)
            # print(water_fraction)f cloud_arr[track_number-1] is None:
            
            if not (self.track_start_time):
                self.track_start_time = time
                self.n_timesteps = 1
            else:
                self.n_timesteps += 1
            if self.n_timesteps <= 4:
                self.start_ice_fraction_arr[self.n_timesteps-1] = ice_fraction
            # Check and set type of cloud
            if water_fraction > 1-self.crit_fraction:
                self.is_liq = True
            elif water_fraction > self.crit_fraction:
                self.is_mix = True
            else:
                self.is_ice = True
            if self.is_resampled:
                cloud_size_km = pixel_area*cloud_size_px * \
                    np.cos(np.deg2rad(cloud_lat))*111.321*111.111
            else:
                cloud_size_km = pixel_area.sum()
                large_pixel_frac = np.count_nonzero(pixel_area>66)/pixel_area.shape[0]
                if large_pixel_frac>0.1 or pixel_area.max()>110:
                    self.large_pixel_cloud = True
            self.cloud_size_km_list.append(cloud_size_km)
            self.max_size_km = max(self.max_size_km, cloud_size_km)
            self.min_size_km = min(self.min_size_km, cloud_size_km)

            self.max_size_px = max(self.max_size_px, cloud_size_px)
            self.min_size_px = min(self.min_size_px, cloud_size_px)

            self.sum_cloud_size_px += cloud_size_px
            self.avg_cloud_size_px = self.sum_cloud_size_px/self.n_timesteps

            self.sum_cloud_size_km += cloud_size_km
            self.avg_cloud_size_km = self.sum_cloud_size_km/self.n_timesteps

            # I assume that water_frac+ice_frac=1

            self.max_water_fraction = max(
                self.max_water_fraction, water_fraction)
            self.max_ice_fraction = max(
                self.max_ice_fraction, 1-water_fraction)

            self.sum_cloud_lat += cloud_lat
            self.sum_cloud_lon += cloud_lon
            self.lon_list.append(cloud_lon)
            self.lat_list.append(cloud_lat)
            self.avg_cloud_lat = self.sum_cloud_lat/self.n_timesteps
            self.avg_cloud_lon = self.sum_cloud_lon/self.n_timesteps

            self.track_end_time = time
            self.track_length = self.track_end_time-self.track_start_time

            self.end_ice_fraction_arr[0:3] = self.end_ice_fraction_arr[1:4]
            self.end_ice_fraction_arr[3] = ice_fraction

            # self.ice_fraction_arr[n_timesteps]=ice_fraction
            self.ice_fraction_list.append(ice_fraction)

            self.update_cot_variables(cot_values,pixel_area)
            self.update_ctp_variables(ctp_values,pixel_area)
            
    def update_cot_variables(self,cot_values,pixel_area):
        cot_nan_frac = np.count_nonzero(np.isnan(cot_values))/cot_values.shape[0]
        if cot_nan_frac>0.1:
            self.valid_cot_cloud=False
        self.cot_nan_frac_list.append(cot_nan_frac)
        weights = pixel_area[~np.isnan(cot_values)]
        if len(weights)>0:
            cot_values = cot_values[~np.isnan(cot_values)]
            mean_cot = np.average(cot_values,weights=weights)
            if cot_nan_frac<0.1:
                self.sum_cloud_cot+=mean_cot
                self.cot_timestep_counter+=1
                self.avg_cot=self.sum_cloud_cot/self.cot_timestep_counter
        else:
            mean_cot = np.nan
        self.mean_cot_list.append(mean_cot)

    def update_ctp_variables(self, ctp_values,pixel_area):
        ctp_nan_frac = np.count_nonzero(np.isnan(ctp_values))/ctp_values.shape[0]
        if ctp_nan_frac>0.1:
            self.valid_ctp_cloud=False
        self.ctp_nan_frac_list.append(ctp_nan_frac)
        weights = pixel_area[~np.isnan(ctp_values)]
        if len(weights)>0:
            ctp_values = ctp_values[~np.isnan(ctp_values)]
            mean_ctp = np.average(ctp_values,weights=weights)
            if ctp_nan_frac<0.1:
                self.sum_cloud_ctp+=mean_ctp
                self.ctp_timestep_counter+=1
                self.avg_ctp=self.sum_cloud_ctp/self.ctp_timestep_counter
        else:
            mean_ctp = np.nan
        self.mean_ctp_list.append(mean_ctp)

    def update_missing_cloud(self):
        if self.track_end_time and (not self.terminate_cloud):
            self.n_timesteps_no_cloud += 1
            if self.n_timesteps_no_cloud > 1:
                self.terminate_cloud = True